# NHS England Maternal Care Pathways Master Pipeline
## Stage 2.1 - Load other MSDS data tables and join to aggregated demographics data

### Compiled by Ethan Phillips (Oxford)

### Last updated 10.11.2025


In [0]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession 
from pyspark.sql.functions import when, col, lit, count, max, last, first, min, avg, sum, collect_list, collect_set, countDistinct, to_date, datediff, array_contains, size, udf, format_number, regexp_replace, explode, array, array_union, desc_nulls_last
from pyspark.sql.types import IntegerType, StringType, NumericType, DateType, ArrayType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.ml.regression import GeneralizedLinearRegression as GLR


In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
read_table_name = "msds_demographics_agg_by_uniqpregid_stage"

df_demographics_agg = spark.read.table(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{read_table_name}")

In [0]:
features_to_keep = list(df_demographics_agg.columns)

In [0]:
df_baby_activities = spark.sql("SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_baby_activities_all_years").select(
    "UniqPregID",
    "NNCritIncInd",
    "ApgarScore",
    "BirthWeight"
).groupby("UniqPregID").agg(
    min("ApgarScore").try_cast("double").alias("apgar_score"),
    min("BirthWeight").try_cast("double").alias("birth_weight"),
    collect_set("NNCritIncInd").alias("neonatal_crit_incident")
)

features_to_keep.extend(["apgar_score", "birth_weight", "neonatal_crit_incident"])

print(f"Number of rows: {'{:,}'.format(df_baby_activities.count())}. Number of columns: {'{:,}'.format(len(df_baby_activities.columns))}")

In [0]:
df_baby_demographic = spark.sql(
    "SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_baby_demographics_all_years"
).select(
    "UniqPregID",
    "AgeAtBirthMother",
    "AgeAtDeathBaby",
    "DeliveryMethodCode",
    "EthnicCategoryBaby",
    "FetusPresentation",
    "GestationLengthBirth",
    "DayOfBirthBaby",
    "MerOfBirthBaby",
    "MonthOfBirthBaby",
    "YearOfBirthBaby",
    "PersonPhenSex",
    "PregOutcome",
    "NeoCritCareInd",
    "OrgSiteIDActualDelivery"
).groupby("UniqPregID").agg(
    max("AgeAtBirthMother").try_cast('int').alias("mother_age_at_birth"),
    min("AgeAtDeathBaby").try_cast('int').alias("baby_age_at_death"),
    last("DeliveryMethodCode", ignorenulls=True).try_cast('int').alias("delivery_method"),
    collect_set("EthnicCategoryBaby").alias("baby_ethnicity"),
    last("FetusPresentation", ignorenulls=True).try_cast('int').alias("fetus_presentation"),
    min("GestationLengthBirth").try_cast('int').alias("gestation_length_at_birth"),

    last("DayOfBirthBaby", ignorenulls=True).try_cast('int').alias("birth_day_of_week"),
    last("MerOfBirthBaby", ignorenulls=True).try_cast('int').alias("birth_am_or_pm"),
    last("MonthOfBirthBaby", ignorenulls=True).try_cast('int').alias("birth_month"),
    last("YearOfBirthBaby", ignorenulls=True).try_cast('int').alias("birth_year"),

    last("PersonPhenSex", ignorenulls=True).try_cast('int').alias("baby_phen_sex"),
    last("PregOutcome", ignorenulls=True).try_cast('int').alias("pregnancy_outcome"),
    collect_set("NeoCritCareInd").alias("neonatal_critical_care"),
    last('OrgSiteIDActualDelivery', ignorenulls=True).alias('ld_site_id'),
)

# Add the newly created feature names to the master list
features_to_keep.extend([
    "mother_age_at_birth",
    "baby_age_at_death",
    "delivery_method",
    "baby_ethnicity",
    "fetus_presentation",
    "gestation_length_at_birth",
    "birth_day_of_week",
    "birth_am_or_pm",
    "birth_month",
    "birth_year",
    "baby_phen_sex",
    "pregnancy_outcome",
    "neonatal_critical_care",
    "ld_site_id"
])

print(f"Number of rows: {'{:,}'.format(df_baby_demographic.count())}. Number of columns: {'{:,}'.format(len(df_baby_demographic.columns))}")

In [0]:
df_care_activities = spark.sql("SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_care_activities_all_years").select(
    "UniqPregID",
    "AlcoholUnitsPerWeek",
    "CigarettesPerDay",
    "PersonHeight",
    "PersonWeight",
    "SmokingStatus"
).groupby("UniqPregID").agg(
    max("AlcoholUnitsPerWeek").try_cast('float').alias("max_alcohol_per_week"),
    first("AlcoholUnitsPerWeek", ignorenulls=True).try_cast('float').alias("first_alcohol_per_week"),
    last("AlcoholUnitsPerWeek", ignorenulls=True).try_cast('float').alias("last_alcohol_per_week"),

    max("CigarettesPerDay").try_cast('int').alias("max_cigs_per_day"),
    first("CigarettesPerDay", ignorenulls=True).try_cast('int').alias("first_cigs_per_day"),
    last("CigarettesPerDay", ignorenulls=True).try_cast('int').alias("last_cigs_per_day"),

    avg("PersonHeight").try_cast('float').alias("mother_avg_height"),

    avg("PersonWeight").try_cast('float').alias("mother_avg_weight"),
    first("PersonWeight", ignorenulls=True).try_cast('float').alias("mother_first_weight"),
    last("PersonWeight", ignorenulls=True).try_cast('float').alias("mother_last_weight"),

    collect_list("SmokingStatus").alias("smoking_status")
)

# Add the newly created care activity feature names to the master list
features_to_keep.extend([
    "max_alcohol_per_week",
    "first_alcohol_per_week",
    "last_alcohol_per_week",
    "max_cigs_per_day",
    "first_cigs_per_day",
    "last_cigs_per_day",
    "mother_avg_height",
    "mother_avg_weight",
    "mother_first_weight",
    "mother_last_weight",
    "smoking_status"
])

print(f"Number of rows: {'{:,}'.format(df_care_activities.count())}. Number of columns: {'{:,}'.format(len(df_care_activities.columns))}")

In [0]:
df_labour = spark.sql("SELECT DISTINCT * FROM dsa_712819_x8g2j.dsa_712819_x8g2j_collab.msds_v2_labour_activities_all_years").select(
    "UniqPregID",
    "MatCritInd"
).groupby("UniqPregID").agg(
    collect_set("MatCritInd").alias("maternal_crit_incident")
)

features_to_keep.append("maternal_crit_incident")

print(f"Number of rows: {'{:,}'.format(df_labour.count())}. Number of columns: {'{:,}'.format(len(df_labour.columns))}")

In [0]:
df_master_allyrs = df_demographics_agg.join(df_baby_activities, on="UniqPregID", how="left").join(df_baby_demographic, on="UniqPregID", how="left").join(df_care_activities, on="UniqPregID", how="left").join(df_labour, on="UniqPregID", how="left")

# non-LD hosp spells left out

print(f"Number of rows: {'{:,}'.format(df_master_allyrs.count())}. Number of columns: {'{:,}'.format(len(df_master_allyrs.columns))}")

In [0]:
df_master_allyrs = df_master_allyrs.select(features_to_keep)

In [0]:
write_table_name = "msds_all_agg_filtered_stage"

df_master_allyrs.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"dsa_712819_x8g2j.dsa_712819_x8g2j_collab.{write_table_name}")